In [ ]:
'''Multithreading in Python is a technique that allows a program to run multiple threads concurrently within the same process. A thread is the smallest unit of execution in a program.
task -> process -> thread
1 task is the overall work to be done.A task is often implemented by one or more processes.
2 process is an instance of a running program.
3 thread is the smallest unit of execution inside a process.

Multiprocessing is a technique where a program creates multiple processes, and each process runs independently with its own memory space.

GIL (Global Interpreter Lock) is a mechanism in CPython (the standard Python interpreter) that allows only one thread to execute Python bytecode at a time, even if multiple threads exist.
'''

In [6]:
from multiprocessing import Process
import os
def calculate():
    total = 0
    for i in range(10_000_000):
        total += i
    print(f"Total = {total}",os.getpid()) #.getpid() gives the process id

if __name__ == "__main__":
    p1 = Process(target=calculate)
    p2 = Process(target=calculate)

    p1.start()
    p2.start()

    p1.join()
    p2.join()  #.join() makes the main process (or main thread) wait until a process (or thread) finishes. or without it print("All processes completed") runs directly

    print("All processes completed")

'''Total = 49999995000000
Total = 49999995000000
All processes completed'''

All processes completed


In [7]:
#Passing Arguments to a Process

from multiprocessing import Process

def square(n):
    print(f"Square of {n} = {n*n}",)

if __name__ == "__main__":
    p1 = Process(target=square, args=(5,))  #arguments as tuples
    p2 = Process(target=square, args=(10,))

    p1.start()
    p2.start()

    p1.join()
    p2.join()

'''Square of 5 = 25
Square of 10 = 100'''

In [8]:
from multiprocessing import Process
import time

def calculate():
    for i in range(10):
        print(i)

if __name__ == "__main__":
    p1 = Process(target=calculate)
    p2 = Process(target=calculate)

    p1.start()
    p2.start()

    p1.join()
    p2.join()

    #is a very small task. The OS may let Process 1 run to completion before switching to Process 2.

In [2]:
#adding a small delay
from multiprocessing import Process
import time

def calculate():
    for i in range(10):
        print(i)
        time.sleep(0.1)

if __name__ == "__main__":
    p1 = Process(target=calculate)
    p2 = Process(target=calculate)

    p1.start()
    p2.start()

    p1.join()
    p2.join()

In [1]:
from threading import Thread
import time

def calculate(name):
    for i in range(5):
        print(f"{name}: {i}")
        time.sleep(1)

if __name__ == "__main__":
    t1 = Thread(target=calculate, args=("Thread-1",))
    t2 = Thread(target=calculate, args=("Thread-2",))

    t1.start()
    t2.start()

    t1.join()
    t2.join()

    print("All threads completed")

Thread-1: 0
Thread-2: 0
Thread-1: 1
Thread-2: 1
Thread-1: 2Thread-2: 2

Thread-2: 3Thread-1: 3

Thread-1: 4
Thread-2: 4
All threads completed


Thread vs Process

Threads
One Process
├── Thread 1
├── Thread 2
└── Thread 3
Share memory
Lightweight
Fast communication
Affected by GIL

Processes
Process 1
Process 2
Process 3

Separate memory
True parallelism
Not limited by a single GIL

In [ ]:
'''In Python, a Pool is a feature from the multiprocessing module that creates a group of worker processes to execute tasks concurrently.
Instead of manually creating and managing multiple processes, a Pool handles them for you'''

from multiprocessing import Pool

def square(n):
    return n * n

if __name__ == "__main__":
    with Pool(4) as pool:
        results = pool.map(square, [1, 2, 3, 4, 5])

    print(results)

In [ ]:
#A Queue is used in multiprocessing to exchange data between processes.
from multiprocessing import Process, Queue

def square(n, q):
    q.put(n * n)

if __name__ == "__main__":
    q = Queue()

    p = Process(target=square, args=(5, q))

    p.start()
    p.join()

    result = q.get()
    print(result) #25

In [ ]:
#Multiple Processes Writing to a Queue
from multiprocessing import Process, Queue

def worker(n, q):
    q.put(n * n)

if __name__ == "__main__":
    q = Queue()

    processes = []

    for i in range(1, 6):
        p = Process(target=worker, args=(i, q))
        processes.append(p)
        p.start()

    for p in processes:
        p.join()

    while not q.empty():
        print(q.get())

1. CPU-Bound
A task is CPU-bound when it spends most of its time performing calculations and processing data. The CPU is the bottleneck.
Examples

Image/video processing
Data compression
Machine learning training
Mathematical simulations
Encryption/decryption

2. I/O-Bound
A task is I/O-bound when it spends most of its time waiting for input/output operations rather than doing computations.
"I/O" stands for Input/Output, such as:

Reading files from disk
Writing files
Database queries
Network requests
API calls

In [ ]:
'''1. User-Level Threads (User-Defined Threads)
These threads are managed by a user-space thread library rather than directly by the operating system kernel.
Features

Created and managed by the application or runtime.
Kernel is not aware of these threads.
Faster to create and switch between threads.
If one thread blocks on a system call, the entire process may block.

Advantages
✅ Fast thread creation
✅ Fast context switching
✅ Less kernel overhead
Disadvantages
❌ A blocking system call can block all threads in the process
❌ Cannot fully utilize multiple CPUs unless supported by the threading library
Example

Green Threads (older Java implementations)
User-level thread libraries


2. Kernel Threads
These threads are directly managed by the Operating System Kernel.
Features

Kernel knows about every thread.
Each thread can be scheduled independently.
Multiple threads can run simultaneously on multiple CPU cores.

Advantages
✅ Better concurrency
✅ True parallel execution on multicore systems
✅ One blocked thread doesn't stop others
Disadvantages
❌ Creation and context switching are slower than user threads
❌ Higher OS overhead
Examples

POSIX Threads (Pthreads) in Linux
Windows Threads
Modern Java Threads


3. Worker Threads
A worker thread is a thread that performs background work on behalf of the main thread.
Features

Executes tasks asynchronously.
Often taken from a thread pool.
Prevents the main thread from being blocked.

Uses

Processing user requests
File operations
Database queries
Background computations

Example
Plain TextMain Thread     |     +--> Worker Thread 1 (process file)     +--> Worker Thread 2 (send email)     +--> Worker Thread 3 (database query)Show more lines
Benefits
✅ Improves responsiveness
✅ Allows concurrent task execution

4. Daemon Threads
A daemon thread runs in the background and supports other threads.
Features

Provides services to user threads.
Terminates automatically when all non-daemon threads finish.

Examples

Garbage Collector in Java
Background monitoring services
Logging services

Java Example
JavaThread t = new Thread();t.setDaemon(true);Show more lines
Characteristics
✅ Background tasks
✅ Doesn't prevent program termination

5. Non-Daemon Threads (User Threads)
These are regular threads that perform the main work of an application.
Features

Keep the application alive.
Program exits only after all non-daemon threads finish.

Examples

Main thread
Request-handling threads
Calculation threads

Characteristics
✅ Essential application work
✅ Must complete before process exits

Threads
│
├── User-Level Threads
│   └── Managed by thread library
│
├── Kernel Threads
│   └── Managed by OS kernel
│
├── Worker Threads
│   └── Perform background tasks
│
└── Based on Lifecycle
    ├── Daemon Threads
    │   └── Background service threads
    │
    └── Non-Daemon Threads
        └── Main/important application threads'''